# HARMPI GRMHD: SANE and MAD candidate configurations
Adapted from Robert Wilson Joute's uploaded notebook. CPU Colab; fresh runtime required. This edited version preserves the physics parameters but improves build checks, waits for completion and preserves baseline dumps. Not execution validated.

## 1. Check runtime

In [ ]:
!nproc
!gcc --version | head -1
!python --version

## 2. Clone pinned upstream revision

In [ ]:
%cd /content
!git clone -q https://github.com/atchekho/harmpi.git
%cd /content/harmpi
!git checkout -q d2673a9
!git log --oneline -1
!ls

## 3. Inspect original build

In [ ]:
!make 2>&1 | tail -n 2
!ls -l harm

## 4. Inspect torus setup

In [ ]:
!sed -n '82p;115,118p' decs.h
!sed -n '73p' init.c
!sed -n '151,169p' init.c
!sed -n '213,215p' init.c
!sed -n '405,416p' init.c

## 5. Inspect numerical methods

In [ ]:
!sed -n '57,61p' step_ch.c
!sed -n '432,446p' step_ch.c
!sed -n '163,165p' init.c
!sed -n '94p;111p' step_ch.c
!sed -n '204p;246p' step_ch.c

## 6. Checked patch helper

In [ ]:
def setline(fname, n, new, expect):
    """Replace line n after checking the old line contains expect."""
    lines = open(fname).read().split("\n")
    old = lines[n - 1]
    if expect not in old:
        raise ValueError(f"{fname}:{n} reads {old.strip()!r}, which does not contain {expect!r}. Wrong line?")
    indent = old[:len(old) - len(old.lstrip())]
    lines[n - 1] = indent + new
    open(fname, "w").write("\n".join(lines))
    print(f"{fname}:{n}\n   old: {old.strip()}\n   new: {new}")

## 7. Configure baseline

In [ ]:
setline("decs.h",  116, "#define N1       (64)", expect="define N1")
setline("decs.h",  117, "#define N2       (64)", expect="define N2")
setline("init.c",   73, "#define WHICH_FIELD_NORMALIZATION NORMALIZE_FIELD_BY_MAX_RATIO", expect="WHICH_FIELD_NORMALIZATION")
setline("init.c",  151, "gam = 13./9. ;",  expect="gam =")
setline("init.c",  154, "a = 0.9375 ;",    expect=" a =")
setline("init.c",  155, "rin = 10. ;",     expect="rin =")
setline("init.c",  156, "rmax = 20. ;",   expect="rmax =")
setline("init.c",  169, "Rout = 1000. ;",  expect="Rout =")
setline("init.c",  213, "tf = 200. ;",     expect="tf =")
setline("restart.c", 299, "tf = 200. ;",   expect="tf =")

## 8. Review source changes

In [ ]:
!git diff --stat
!git diff init.c | grep "^[-+] "

## 9. Compile with error checking

In [ ]:
import subprocess
from pathlib import Path
subprocess.run(["make", "clean"], check=True)
subprocess.run(["make"], check=True)
assert Path("harm").is_file()

## 10. Run short baseline

In [ ]:
import shutil
# Only delete dumps in this dedicated clone.
assert Path.cwd() == Path("/content/harmpi")
if Path("dumps").exists():
    shutil.rmtree("dumps")
with open("run.log","w") as log:
    subprocess.run(["./harm","1","1","1"], stdout=log, stderr=subprocess.STDOUT, check=True)
!grep -E "rhomax|beta" run.log
!tail -n 2 run.log
!ls dumps

## 11. Load analysis tools

In [ ]:
%matplotlib inline
%run -i harm_script.py
plt.rcdefaults()
rg("gdump")
print("grid:", nx, "x", ny, "x", nz, "  spin a =", a, "  r from", r.min().round(2), "to", r.max().round(0))

## 12. Density and magnetic contours

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 5), sharey=True)
for ax, name in zip(axs, ["dump000", "dump020"]):
    rd(name)
    X, Z = r*np.sin(h), r*np.cos(h)
    plt.sca(ax)
    plc(np.log10(rho), xcoord=X, ycoord=Z, isfilled=True, cb=True, levels=np.linspace(-8, 0, 81))
    plc(psicalc(), xcoord=X, ycoord=Z, levels=15, colors="w", linewidths=0.6)
    ax.set_xlim(0, 40); ax.set_ylim(-20, 20); ax.set_aspect("equal")
    ax.set_title(f"log10 rho and field lines, t = {t:g} M")
plt.show()

## 13. Plasma beta and magnetization

In [ ]:
rd("dump020")
beta  = 2*pg/(bsq + 1e-30)
sigma = bsq/rho + 1e-12
X, Z = r*np.sin(h), r*np.cos(h)
fig, axs = plt.subplots(1, 2, figsize=(10, 5), sharey=True)
plt.sca(axs[0]); plc(np.log10(beta),  xcoord=X, ycoord=Z, isfilled=True, cb=True, levels=np.linspace(-1, 4, 51)); axs[0].set_title("log10 beta")
plt.sca(axs[1]); plc(np.log10(sigma), xcoord=X, ycoord=Z, isfilled=True, cb=True, levels=np.linspace(-6, 2, 41)); axs[1].set_title("log10 sigma")
for ax in axs: ax.set_xlim(0, 40); ax.set_ylim(-20, 20); ax.set_aspect("equal")
plt.show()

## 14. Horizon flux diagnostics

In [ ]:
import glob, os
def horizon_fluxes(rundir):
    """Mdot, Phi_BH and phi_BH = Phi/sqrt(Mdot) at the horizon."""
    here = os.getcwd(); os.chdir(rundir)
    try:
        rg("gdump")
        ih = iofr(1 + np.sqrt(1 - a**2))
        rows = []
        for f in sorted(glob.glob("dumps/dump[0-9][0-9][0-9]")):
            rd(os.path.basename(f))
            Mdot = -(gdet*rho*uu[1])[ih].sum()*_dx2*_dx3
            Phi  = 0.5*np.abs(gdet*B[1])[ih].sum()*_dx2*_dx3
            rows.append((t, Mdot, Phi, Phi/np.sqrt(abs(Mdot)) if Mdot != 0 else np.nan))
        if not rows:
            raise RuntimeError("No numbered dumps found")
        return np.array(rows)
    finally:
        os.chdir(here)

F = horizon_fluxes("/content/harmpi")
fig, axs = plt.subplots(1, 2, figsize=(11, 3.5))
axs[0].plot(F[:, 0], F[:, 1]); axs[0].set_xlabel("t [M]"); axs[0].set_ylabel("Mdot")
axs[1].plot(F[:, 0], F[:, 3]); axs[1].set_xlabel("t [M]"); axs[1].set_ylabel("phi_BH")
plt.tight_layout(); plt.show()

## 15. Prepare comparison directories

In [ ]:
%cd /content/harmpi
setline("init.c",    213, "tf = 1000. ;", expect="tf =")
setline("restart.c", 299, "tf = 1000. ;", expect="tf =")
subprocess.run(["make"], check=True)
# Preserve baseline and clone before starting either long run.
if Path("/content/harmpi_MAD").exists():
    raise FileExistsError("Use a fresh runtime or archive the existing MAD run")
if Path("dumps").exists():
    Path("dumps").rename("baseline_dumps")
if Path("run.log").exists():
    Path("run.log").rename("baseline.log")
shutil.copytree("/content/harmpi", "/content/harmpi_MAD")

## 16. Configure alternate field and launch

In [ ]:
%cd /content/harmpi_MAD
setline("init.c", 412, "q = rho_av/rhomax*pow(r/rin,3.)*pow(sin(th),3.)*exp(-r/400.) ;", expect="q =")
subprocess.run(["make","clean"],check=True)
subprocess.run(["make"],check=True)
def launch(directory):
    with open(Path(directory)/"run.log","w") as log:
        return subprocess.Popen(["./harm","1","1","1"], cwd=directory, stdout=log, stderr=subprocess.STDOUT)
sane = launch("/content/harmpi")
mad = launch("/content/harmpi_MAD")

## 17. Monitor and wait for completion

In [ ]:
print("SANE running:", sane.poll() is None, "   MAD running:", mad.poll() is None)
!grep -E "initial beta|final beta" /content/harmpi/run.log /content/harmpi_MAD/run.log
for proc in (sane,mad):
    if proc.wait() != 0:
        raise RuntimeError("Simulation failed; inspect run.log")

## 18. Compare fluxes

In [ ]:
S, M = horizon_fluxes("/content/harmpi"), horizon_fluxes("/content/harmpi_MAD")
fig, axs = plt.subplots(1, 2, figsize=(11, 3.5))
for F, lab in [(S, "SANE"), (M, "MAD")]:
    axs[0].plot(F[:, 0], F[:, 1], label=lab); axs[1].plot(F[:, 0], F[:, 3], label=lab)
axs[0].set_ylabel("Mdot"); axs[1].set_ylabel("phi_BH")
for ax in axs: ax.set_xlabel("t [M]"); ax.legend()
plt.tight_layout(); plt.show()

## 19. Optional Google Drive export

In [ ]:
# Optional: run only if you want to export to Google Drive.
from google.colab import drive
drive.mount("/content/drive")
DEST = Path("/content/drive/MyDrive/harmpi_T4")
for run, src in [("SANE", "/content/harmpi"), ("MAD", "/content/harmpi_MAD")]:
    dest = DEST/run
    (dest/"dumps").mkdir(parents=True, exist_ok=True)
    for f in (Path(src)/"dumps").glob("*"):
        if f.is_file():
            shutil.copy2(f, dest/"dumps")
    shutil.copy2(Path(src)/"run.log",dest)
    with open(dest/"changes.diff","w") as out:
        subprocess.run(["git","diff"],cwd=src,stdout=out,check=True)